# Multi-dataset T5 entity reconstruction

This notebook fine-tunes a T5 encoder-decoder on several masked summarization datasets. Each occurrence of `<mask>` is converted to a native numbered T5 sentinel (`<extra_id_0>`, `<extra_id_1>`, ...), and the decoder generates only the ordered missing entity spans. Training and validation loss are written to a JSONL log, periodic checkpoints are saved, and the final model is exported.

In [1]:
# %pip install torch transformers datasets sentencepiece

import json
import math
import random
import re
import time
from pathlib import Path
import gc
import torch

import torch
from datasets import concatenate_datasets, load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

W0901 16:07:30.314000 25564 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
#from huggingface_hub import login
#login()

In [3]:
DATASET_PATHS = {
    "cnn_dailymail": "data/cnn_dailymail_with_mask",
    "xsum": "data/xsum_with_mask",
    "samsum": "data/samsum_with_mask",
    "multi_news": "data/multi_news_with_mask",
    "billsum": "data/billsum_with_mask",
}
MODEL_NAME = "google/flan-t5-base"
RUN_DIR = Path("models/flan-t5-base-2k")

# CNN/DailyMail is dominating.
# None to use every available example.
MAX_TRAIN_PER_DATASET = 100_000
MAX_EVAL_PER_DATASET = 200

MAX_INPUT_LENGTH = 2048
MAX_SENTINELS = 100
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 16
EPOCHS = 1
LEARNING_RATE = 5e-5 
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.02
MAX_GRAD_NORM = 1.0
LOG_EVERY_STEPS = 100
EVAL_EVERY_STEPS = 500 
SAVE_EVERY_STEPS = 1000
SEED = 42
USE_AMP = True
RESUME_FROM = None

RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / "loss_log.jsonl"

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = USE_AMP and device.type == "cuda"

if amp_enabled and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
elif amp_enabled:
    amp_dtype = torch.float16
else:
    amp_dtype = torch.bfloat16
scaler_enabled = amp_enabled and amp_dtype == torch.float16
print(
    f"device={device}, mixed_precision={amp_enabled}, "
    f"dtype={amp_dtype}, grad_scaler={scaler_enabled}"
)

tokenizer_source = RESUME_FROM if RESUME_FROM else MODEL_NAME
model_source = RESUME_FROM if RESUME_FROM else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_source,
    torch_dtype='bfloat16'
).to(device)
model.gradient_checkpointing_enable()
model.config.use_cache = False


sentinel_tokens = [f"<extra_id_{i}>" for i in range(MAX_SENTINELS)]
sentinel_ids = tokenizer.convert_tokens_to_ids(sentinel_tokens)
invalid_sentinels = [
    token
    for token, token_id in zip(sentinel_tokens, sentinel_ids)
    if token_id is None or token_id == tokenizer.unk_token_id
]

if invalid_sentinels or len(set(sentinel_ids)) != MAX_SENTINELS:
    raise NotImplementedError(f"Tokenizer is missing {len(invalid_sentinels)} numbered T5 sentinels: {invalid_sentinels}.")


print(f"Validated {len(sentinel_ids)} numbered T5 sentinels")

device=cuda, mixed_precision=True, dtype=torch.bfloat16, grad_scaler=False


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Validated 100 numbered T5 sentinels


## Number masks and build T5 targets

The original files remain unchanged. Numbering happens in memory. An example containing $m$ masks becomes:

```text
input:  <extra_id_0> ... <extra_id_1> ...
target: <extra_id_0> entity 0 <extra_id_1> entity 1 ...
```

In [5]:
MASK_PATTERN = re.compile(r"<mask>")


def number_masks(masked_text, entities):
    mask_count = len(MASK_PATTERN.findall(masked_text))
    if mask_count != len(entities):
        raise ValueError(
            f"Found {mask_count} masks but {len(entities)} targets"
        )
    
    if mask_count == 0 or mask_count > MAX_SENTINELS:
        raise ValueError(f"Unsupported mask count: {mask_count}")

    index = 0
    def replace_mask(_match):
        nonlocal index
        sentinel = f"<extra_id_{index}>"
        index += 1
        return sentinel

    numbered_document = MASK_PATTERN.sub(replace_mask, masked_text)
    target = " ".join(
        f"<extra_id_{i}> {entity}"
        for i, entity in enumerate(entities)
    )
    return numbered_document, target


def make_texts(row):
    document, target = number_masks(
        row["masked_text"], row["demasked_words"]
    )
    source = f"summary: {row['summary']}\nmasked document: {document}"
    return source, target

In [6]:
def select_examples(split, limit, seed):
    split = split.shuffle(seed=seed)
    if limit is not None:
        split = split.select(range(min(limit, len(split))))
    return split


def prepare_raw_split(split, dataset_name):
    required = {"summary", "masked_text", "demasked_words"}
    missing = required - set(split.column_names)
    if missing:
        raise ValueError(f"{dataset_name} is missing columns: {missing}")

    def valid(row):
        try:
            source, target = make_texts(row)
        except (TypeError, ValueError):
            return False
        source_length = len(tokenizer(
            source,
            truncation=True,
            max_length=MAX_INPUT_LENGTH + 1,
        )["input_ids"])
        return source_length <= MAX_INPUT_LENGTH

    split = split.filter(
        valid, desc=f"Validating {dataset_name}"
    )
    return split.select_columns([
        "summary", "masked_text", "demasked_words"
    ])


train_parts = []
eval_parts = []
dataset_counts = {}

for offset, (name, path) in enumerate(DATASET_PATHS.items()):
    disk_dataset = load_from_disk(path)
    eval_name = "validation" if "validation" in disk_dataset else "test"

    train_part = select_examples(
        disk_dataset["train"], MAX_TRAIN_PER_DATASET, SEED + offset
    )
    eval_part = select_examples(
        disk_dataset[eval_name], MAX_EVAL_PER_DATASET, SEED + offset
    )
    train_part = prepare_raw_split(train_part, f"{name}/train")
    eval_part = prepare_raw_split(eval_part, f"{name}/{eval_name}")

    dataset_counts[name] = {
        "train": len(train_part),
        "eval": len(eval_part),
    }
    train_parts.append(train_part)
    eval_parts.append(eval_part)

train_raw = concatenate_datasets(train_parts).shuffle(seed=SEED)
eval_raw = concatenate_datasets(eval_parts).shuffle(seed=SEED)
print(json.dumps(dataset_counts, indent=2))
print(f"combined train={len(train_raw):,}, eval={len(eval_raw):,}")

Validating cnn_dailymail/train:   0%|          | 0/100000 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
next(model.parameters()).dtype

torch.bfloat16

In [ ]:
def tokenize_row(row):
    source, target = make_texts(row)
    encoded = tokenizer(
        source, max_length=MAX_INPUT_LENGTH, truncation=True
    )
    target_tokens = tokenizer(text_target=target)
    encoded["labels"] = target_tokens["input_ids"]
    return encoded


train_set = train_raw.map(
    tokenize_row,
    remove_columns=train_raw.column_names,
    desc="Tokenizing train data",
)
eval_set = eval_raw.map(
    tokenize_row,
    remove_columns=eval_raw.column_names,
    desc="Tokenizing eval data",
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, label_pad_token_id=-100
)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    pin_memory=device.type == "cuda",
)
eval_loader = DataLoader(
    eval_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    pin_memory=device.type == "cuda",
)

diagnostic_batch = next(iter(train_loader))
input_ids = diagnostic_batch["input_ids"]
labels = diagnostic_batch["labels"]
assert input_ids.min().item() >= 0
valid_labels = labels[labels != -100]
assert valid_labels.numel() > 0
assert valid_labels.min().item() >= 0
assert len(tokenizer) <= model.get_input_embeddings().num_embeddings
assert max(sentinel_ids) < model.get_input_embeddings().num_embeddings

model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type, enabled=amp_enabled, dtype=amp_dtype
):
    diagnostic_loss = model(**{
        key: value.to(device) for key, value in diagnostic_batch.items()
    }).loss
model.train()
if not torch.isfinite(diagnostic_loss):
    raise FloatingPointError(
        f"Initial forward loss is {diagnostic_loss.item()} with {amp_dtype}. "
        "Set USE_AMP=False if BF16 is unavailable."
    )
print(
    f"diagnostic_loss={diagnostic_loss.item():.4f}, "
    f"input_shape={tuple(input_ids.shape)}, labels={valid_labels.numel()}"
)

Tokenizing train data:   0%|          | 0/237661 [00:00<?, ? examples/s]

Tokenizing eval data:   0%|          | 0/797 [00:00<?, ? examples/s]

diagnostic_loss=2.6182, input_shape=(2, 1477), labels=253


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION)
total_updates = updates_per_epoch * EPOCHS
warmup_steps = int(total_updates * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_updates,
)
scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
global_step = 0
start_epoch = 0

if RESUME_FROM:
    state_path = Path(RESUME_FROM) / "training_state.pt"
    if state_path.exists():
        state = torch.load(state_path, map_location=device)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        if state.get("scaler") is not None:
            scaler.load_state_dict(state["scaler"])
        global_step = state["global_step"]
        start_epoch = state["epoch"]
        print(f"Resumed from epoch {state['epoch']}, step {global_step}")

print(
    f"updates_per_epoch={updates_per_epoch:,}, "
    f"total_updates={total_updates:,}, warmup={warmup_steps:,}"
)

C:\Users\n2bssd\AppData\Local\Temp\ipykernel_31072\424091060.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)


Resumed from epoch 0, step 4500
updates_per_epoch=7,427, total_updates=7,427, warmup=148


In [ ]:
def append_log(record):
    record = {"time": time.time(), **record}
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record) + "\n")


@torch.no_grad()
def evaluate():
    model.eval()
    weighted_loss = 0.0
    target_tokens = 0
    for batch in eval_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        count = (batch["labels"] != -100).sum().item()
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation loss")
        weighted_loss += loss.item() * count
        target_tokens += count
    model.train()
    return weighted_loss / max(target_tokens, 1)


def save_checkpoint(epoch, step):
    checkpoint = RUN_DIR / f"checkpoint-step-{step}"
    checkpoint.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    torch.save(
        {
            "epoch": epoch,
            "global_step": step,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler_enabled else None,
            "config": {
                "model": MODEL_NAME,
                "dataset_paths": DATASET_PATHS,
                "max_input_length": MAX_INPUT_LENGTH,
            },
        },
        checkpoint / "training_state.pt",
    )
    print(f"saved checkpoint: {checkpoint}")

In [ ]:
optimizer.zero_grad(set_to_none=True)
interval_loss = 0.0
interval_microbatches = 0

for epoch in range(start_epoch, EPOCHS):
    model.train()
    for micro_step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            raw_loss = model(**batch).loss
            loss = raw_loss / GRADIENT_ACCUMULATION

        if not torch.isfinite(raw_loss):
            valid_labels = (batch["labels"] != -100).sum().item()
            raise FloatingPointError(
                f"Non-finite loss at epoch={epoch}, micro_step={micro_step}, "
                f"input_shape={tuple(batch['input_ids'].shape)}, "
                f"valid_labels={valid_labels}, dtype={amp_dtype}. "
                "Restart the kernel from a clean checkpoint; if BF16 is "
                "unavailable, set USE_AMP=False."
            )

        scaler.scale(loss).backward()
        interval_loss += raw_loss.item()
        interval_microbatches += 1

        should_update = (
            micro_step % GRADIENT_ACCUMULATION == 0
            or micro_step == len(train_loader)
        )
        if not should_update:
            continue

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        if global_step % LOG_EVERY_STEPS == 0:
            train_loss = interval_loss / max(interval_microbatches, 1)
            record = {
                "split": "train",
                "epoch": epoch,
                "step": global_step,
                "loss": train_loss,
                "learning_rate": scheduler.get_last_lr()[0],
            }
            append_log(record)
            print(record)
            interval_loss = 0.0
            interval_microbatches = 0

        if global_step % EVAL_EVERY_STEPS == 0:
            eval_loss = evaluate()
            record = {
                "split": "eval",
                "epoch": epoch,
                "step": global_step,
                "loss": eval_loss,
                "perplexity": math.exp(min(eval_loss, 20)),
            }
            append_log(record)
            print(record)

        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(epoch, global_step)

    eval_loss = evaluate()
    record = {
        "split": "eval_epoch",
        "epoch": epoch,
        "step": global_step,
        "loss": eval_loss,
        "perplexity": math.exp(min(eval_loss, 20)),
    }
    append_log(record)
    print(record)
    save_checkpoint(epoch, global_step)

{'split': 'train', 'epoch': 0, 'step': 4600, 'loss': 1.8159542153961956, 'learning_rate': 1.941887621926089e-05}
{'split': 'train', 'epoch': 0, 'step': 4700, 'loss': 1.8255889432877301, 'learning_rate': 1.8731968677016073e-05}
{'split': 'train', 'epoch': 0, 'step': 4800, 'loss': 1.8163511731103064, 'learning_rate': 1.804506113477126e-05}
{'split': 'train', 'epoch': 0, 'step': 4900, 'loss': 1.8464940530247986, 'learning_rate': 1.7358153592526446e-05}
{'split': 'train', 'epoch': 0, 'step': 5000, 'loss': 1.8238149156421424, 'learning_rate': 1.6671246050281634e-05}
{'split': 'eval', 'epoch': 0, 'step': 5000, 'loss': 1.6735788561907465, 'perplexity': 5.331213342093275}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved checkpoint: models\flan-t5-large\checkpoint-step-5000
{'split': 'train', 'epoch': 0, 'step': 5100, 'loss': 1.851281765429303, 'learning_rate': 1.598433850803682e-05}
{'split': 'train', 'epoch': 0, 'step': 5200, 'loss': 1.8327475296054034, 'learning_rate': 1.5297430965792007e-05}
{'split': 'train', 'epoch': 0, 'step': 5300, 'loss': 1.8243029689230026, 'learning_rate': 1.461052342354719e-05}
{'split': 'train', 'epoch': 0, 'step': 5400, 'loss': 1.8650452105607838, 'learning_rate': 1.3923615881302377e-05}
{'split': 'train', 'epoch': 0, 'step': 5500, 'loss': 1.839311129245907, 'learning_rate': 1.3236708339057563e-05}
{'split': 'eval', 'epoch': 0, 'step': 5500, 'loss': 1.6712634227021665, 'perplexity': 5.318883552099129}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved checkpoint: models\flan-t5-large\checkpoint-step-5500


In [ ]:
FINAL_DIR = RUN_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Final model saved to {FINAL_DIR.resolve()}")
print(f"Loss log saved to {LOG_PATH.resolve()}")

In [ ]:
if LOG_PATH.exists():
    loss_history = [
        json.loads(line)
        for line in LOG_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print(loss_history[-10:])
else:
    print("No loss log exists yet.")